# SentimentSense: IMDb Movie Review Classifier 🎬

Welcome to the SentimentSense project! In this notebook, we build a complete pipeline to classify movie reviews as positive or negative using the IMDb dataset.

## 1. Setup and Data Loading
First, we install the necessary libraries and download our dataset from Kaggle.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import kagglehub

# Download NLTK data
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
# Download dataset using kagglehub
# Dataset: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

# Check for CSV in the download path
files = os.listdir(path)
csv_file = [f for f in files if f.endswith(".csv")][0]
csv_path = os.path.join(path, csv_file)
df = pd.read_csv(csv_path)

print(f"Dataset loaded with {len(df)} reviews.")
df.head()

## 2. Text Preprocessing

We clean the text by:
1. Removing HTML tags.
2. Lowercasing.
3. Removing special characters and punctuation.
4. Removing stopwords.
5. Applying Lemmatization.

In [ ]:

from tqdm import tqdm
tqdm.pandas()

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Remove HTML tags
    text = re.sub('<.*?>', '', text)
    # Lowercase
    text = text.lower()
    # Remove non-alphabetic characters
    text = re.sub('[^a-zA-Z]', ' ', text)
    # Tokenize and remove stopwords + lemmatize
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(cleaned_words)

# For speed during demonstration, we sample 5000 reviews
df_sample = df.sample(5000, random_state=42).reset_index(drop=True)
print("Cleaning reviews...")
df_sample['cleaned_review'] = df_sample['review'].progress_apply(clean_text)
df_sample.head()


## 3. Exploratory Data Analysis (EDA)
We visualize the class balance, review lengths, and generate WordClouds to see the most frequent words.

In [ ]:

plt.figure(figsize=(6, 4))
sns.countplot(x='sentiment', data=df_sample, palette='viridis')
plt.title('Sentiment Distribution (Sample)')
plt.show()

# Review Length
df_sample['review_len'] = df_sample['cleaned_review'].apply(len)
plt.figure(figsize=(10, 6))
sns.histplot(df_sample[df_sample['sentiment']=='positive']['review_len'], color='green', label='Positive', kde=True)
sns.histplot(df_sample[df_sample['sentiment']=='negative']['review_len'], color='red', label='Negative', kde=True)
plt.title('Review Length Distribution')
plt.legend()
plt.show()


In [ ]:

from wordcloud import WordCloud

def generate_wordcloud(text, title):
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(title)
    plt.axis('off')
    plt.show()

pos_text = ' '.join(df_sample[df_sample['sentiment']=='positive']['cleaned_review'])
neg_text = ' '.join(df_sample[df_sample['sentiment']=='negative']['cleaned_review'])

generate_wordcloud(pos_text, 'Positive Reviews WordCloud')
generate_wordcloud(neg_text, 'Negative Reviews WordCloud')


## 4. Modeling & Evaluation
We split the data, vectorize the text using TF-IDF, and train two models: Naive Bayes and Logistic Regression.

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

# Encode Labels
df_sample['label'] = df_sample['sentiment'].map({'positive': 1, 'negative': 0})

X = df_sample['cleaned_review']
y = df_sample['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF features: {X_train_tfidf.shape[1]}')


In [ ]:

# Model 1: Naive Bayes (Baseline)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

print('Naive Bayes Results:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}')
print(classification_report(y_test, y_pred_nb))


In [ ]:

# Model 2: Logistic Regression (Improved)
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

print('Logistic Regression Results:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print(classification_report(y_test, y_pred_lr))


In [ ]:

# Visualizing Results: Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

# ROC Curve
y_prob_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob_lr)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()
